# N-gram Text Generation from Abraham Lincoln's Speech

This notebook trains a 4-Gram language model on the Abraham Lincoln corpus and generates new sentences that follow the style of the original speech. Because the corpus is a real historical text, the model learns formal and rhetorical patterns rather than random words.

Corpus Source: https://www.gutenberg.org/ebooks/2657

In [11]:
from pathlib import Path
import random
import re
import nltk
import numpy as np
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Ensure NLTK components are ready
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

possible_corpus_paths = [
    Path.cwd() / 'corpus.txt',
    Path.cwd() / 'Assignment_N-Gram' / 'corpus.txt',
]
corpus_path = next((path for path in possible_corpus_paths if path.exists()), None)
if corpus_path is None:
    raise FileNotFoundError('Could not find corpus.txt in the workspace root or Assignment_N-Gram folder.')

notebook_dir = corpus_path.parent
text = corpus_path.read_text(encoding='utf-8')

print('Notebook directory:', notebook_dir)
print('Corpus file:', corpus_path)
print('Corpus exists:', corpus_path.exists())
print('\nCorpus content preview:\n')
print(text[:1000])

Notebook directory: c:\Users\USER\OneDrive\My Codes\GitHub\NLP\Assignment_N-Gram
Corpus file: c:\Users\USER\OneDrive\My Codes\GitHub\NLP\Assignment_N-Gram\corpus.txt
Corpus exists: True

Corpus content preview:

THE PAPERS AND WRITINGS OF ABRAHAM LINCOLN

VOLUME FIVE

CONSTITUTIONAL EDITION

By Abraham Lincoln


Edited by Arthur Brooks Lapsley





THE WRITINGS OF ABRAHAM LINCOLN, Volume Five, 1858-1862




TO SYDNEY SPRING, GRAYVILLE, ILL.

SPRINGFIELD, June 19, 1858.

SYDNEY SPRING, Esq.

MY DEAR SIR:--Your letter introducing Mr. Faree was duly received. There
was no opening to nominate him for Superintendent of Public Instruction,
but through him Egypt made a most valuable contribution to the convention.
I think it may be fairly said that he came off the lion of the day--or
rather of the night. Can you not elect him to the Legislature? It seems to
me he would be hard to beat. What objection could be made to him? What is
your Senator Martin saying and doing? What is Webb about?

Plea

## 2. Text Preprocessing

Before building any representation or language model, the corpus is normalized. The pipeline lowercases the text, removes HTML tags and punctuation, tokenizes the words, optionally removes stop words, and lemmatizes the final tokens.

The preprocessing pipeline can be summarized as:

$$\text{tokens} = \text{Lemmatize}(\text{RemoveStopWords}(\text{Tokenize}(\text{Clean}(\text{text}))))$$

For the N-gram model, sentence boundaries are also marked with `<s>` and `</s>` so the model can learn where sentences begin and end.

In [12]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


def full_preprocess(text, remove_stopwords=True, use_lemmatization=True):
    """
    Applies the complete preprocessing suite from Lab 1.
    Includes regex cleaning, tokenization, stop word stripping, and lemmatization.
    """
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)
    tokens = [word for word in tokens if len(word) > 1 or word in {'i', 'a'}]

    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [word for word in tokens if word not in stop_words]

    if use_lemmatization:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens


sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]
processed_sentences = []

for sentence in sentences:
    sentence_tokens = full_preprocess(sentence, remove_stopwords=False, use_lemmatization=True)
    if len(sentence_tokens) >= 4:
        processed_sentences.append(['<s>', '<s>', *sentence_tokens, '</s>'])

processed_tokens = [token for sentence in processed_sentences for token in sentence]

print('Sentence count:', len(processed_sentences))
print('Token count:', len(processed_tokens))

if processed_sentences:
    print('First sentence tokens:', processed_sentences[0][:25])
    print('First 30 flattened tokens:', processed_tokens[:30])
else:
    print('No valid sentences were found after preprocessing.')

Sentence count: 4358
Token count: 128830
First sentence tokens: ['<s>', '<s>', 'the', 'paper', 'and', 'writing', 'of', 'abraham', 'lincoln', 'volume', 'five', 'constitutional', 'edition', 'by', 'abraham', 'lincoln', 'edited', 'by', 'arthur', 'brook', 'lapsley', 'the', 'writing', 'of', 'abraham']
First 30 flattened tokens: ['<s>', '<s>', 'the', 'paper', 'and', 'writing', 'of', 'abraham', 'lincoln', 'volume', 'five', 'constitutional', 'edition', 'by', 'abraham', 'lincoln', 'edited', 'by', 'arthur', 'brook', 'lapsley', 'the', 'writing', 'of', 'abraham', 'lincoln', 'volume', 'five', 'to', 'sydney']


## 3. Text Representation: Bag of Words and TF-IDF

Before moving into N-gram language modeling, this notebook also demonstrates two classic vector space representations for text.

### Bag of Words (BoW)
BoW represents each document by the counts of words that appear in it, ignoring word order.

### TF-IDF
TF-IDF increases the weight of words that are important in one document but not overly common across the whole corpus.

The TF-IDF score for a term $t$ in document $d$ is:

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

where:

$$\text{TF}(t, d) = \frac{\text{Count of } t \text{ in } d}{\text{Total terms in } d}$$

and

$$\text{IDF}(t, D) = \log\left(\frac{|D|}{1 + |\{d \in D : t \in d\}|}\right)$$

In [13]:
sample_corpus = [
    'The quick brown fox jumps over the lazy dog.',
    'The lazy dog barks loudly at the moon.',
    'Foxes are quick and foxes are clever animals.'
]

sample_preprocessed_corpus = [full_preprocess(doc) for doc in sample_corpus]
raw_clean_docs = [' '.join(doc) for doc in sample_preprocessed_corpus]

print('--- 1. Bag of Words Representation ---')
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(raw_clean_docs)

print('Vocabulary Found:', bow_vectorizer.get_feature_names_out())
print('BoW Matrix Array:\n', bow_matrix.toarray())

print('\n--- 2. TF-IDF Representation ---')
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(raw_clean_docs)

print('TF-IDF Matrix Array:\n', np.round(tfidf_matrix.toarray(), 3))

--- 1. Bag of Words Representation ---
Vocabulary Found: ['animal' 'bark' 'brown' 'clever' 'dog' 'fox' 'jump' 'lazy' 'loudly'
 'moon' 'quick']
BoW Matrix Array:
 [[0 0 1 0 1 1 1 1 0 0 1]
 [0 1 0 0 1 0 0 1 1 1 0]
 [1 0 0 1 0 2 0 0 0 0 1]]

--- 2. TF-IDF Representation ---
TF-IDF Matrix Array:
 [[0.    0.    0.481 0.    0.366 0.366 0.481 0.366 0.    0.    0.366]
 [0.    0.49  0.    0.    0.373 0.    0.    0.373 0.49  0.49  0.   ]
 [0.452 0.    0.    0.452 0.    0.688 0.    0.    0.    0.    0.344]]


## 4. N-Gram Language Modeling

An N-gram language model estimates the probability of the next word from the previous $n-1$ words.

For a general N-gram, the Maximum Likelihood Estimate is:

$$P(w_i \mid w_{i-n+1}, \dots, w_{i-1}) = \frac{C(w_{i-n+1}, \dots, w_i)}{C(w_{i-n+1}, \dots, w_{i-1})}$$

A bigram model uses one previous word, while a trigram model uses two previous words. In this notebook, the models are trained on the preprocessed Abraham Lincoln corpus with sentence-boundary markers, which helps the generated text stay closer to grammatical sentence structure.

In [14]:
def build_ngram_model(tokens, n=2):
    """Builds an N-gram count table from raw tokens."""
    model = defaultdict(Counter)

    for i in range(len(tokens) - n + 1):
        history = tuple(tokens[i:i + n - 1])
        next_word = tokens[i + n - 1]
        model[history][next_word] += 1

    return model


def print_top_continuations(model, history, top_k=8):
    counter = model.get(history, Counter())
    total = sum(counter.values())

    if not counter or total == 0:
        print(f'No continuation found for {history}')
        return

    for word, count in counter.most_common(top_k):
        probability = count / total
        print(f'P({word} | {" ".join(history)}) = {probability:.3f}')


bigram_model = build_ngram_model(processed_tokens, n=2)
trigram_model = build_ngram_model(processed_tokens, n=3)

print('Bigram examples:')
print_top_continuations(bigram_model, ('the',))
print('\n')
print_top_continuations(bigram_model, ('my',))
print('\n')
print_top_continuations(bigram_model, ('i',))

print('\nTrigram examples:')
print_top_continuations(trigram_model, ('my', 'dear'))
print('\n')
print_top_continuations(trigram_model, ('it', 'is'))
print('\n')
print_top_continuations(trigram_model, ('in', 'the'))

Bigram examples:
P(united | the) = 0.033
P(constitution | the) = 0.020
P(state | the) = 0.020
P(union | the) = 0.020
P(people | the) = 0.020
P(government | the) = 0.019
P(same | the) = 0.016
P(whole | the) = 0.012


P(dear | my) = 0.177
P(friend | my) = 0.044
P(own | my) = 0.035
P(opinion | my) = 0.026
P(hand | my) = 0.026
P(purpose | my) = 0.018
P(thanks | my) = 0.018
P(life | my) = 0.015


P(have | i) = 0.139
P(am | i) = 0.079
P(do | i) = 0.046
P(shall | i) = 0.041
P(think | i) = 0.039
P(will | i) = 0.034
P(can | i) = 0.031
P(should | i) = 0.026

Trigram examples:
P(sir | my dear) = 0.975
P(little | my dear) = 0.012
P(madam | my dear) = 0.012


P(not | it is) = 0.098
P(a | it is) = 0.073
P(to | it is) = 0.043
P(the | it is) = 0.030
P(true | it is) = 0.030
P(wrong | it is) = 0.028
P(in | it is) = 0.020
P(that | it is) = 0.020


P(constitution | in the) = 0.046
P(state | in the) = 0.036
P(territory | in the) = 0.030
P(world | in the) = 0.023
P(federal | in the) = 0.022
P(same | in the)

## 5. Generate New Sentences

Once the N-gram probabilities are learned, the model can generate text one word at a time. At each step, the next word is sampled from the conditional distribution of the current history, and generation stops when the end-of-sentence token is produced.

A greedy next-word prediction can be written as:

$$\hat{w}_t = \arg\max_w P(w \mid h_t)$$

while probabilistic sampling follows:

$$w_t \sim P(w \mid h_t)$$

Using sentence boundaries and a low sampling temperature helps the generated sentences stay closer to the formal style of the Lincoln corpus.

In [ ]:
random.seed(7)
np.random.seed(7)


def choose_next_word(counter, temperature=0.15):
    words = list(counter.keys())
    probabilities = np.array(list(counter.values()), dtype=float)
    probabilities = probabilities / probabilities.sum()

    if temperature > 0:
        scaled = np.log(probabilities + 1e-12) / temperature
        probabilities = np.exp(scaled - scaled.max())
        probabilities = probabilities / probabilities.sum()

    return np.random.choice(words, p=probabilities)


def generate_sentence(max_length=30, temperature=0.15):
    sentence = ['<s>', '<s>']

    while len(sentence) < max_length:
        history = tuple(sentence[-2:])
        counter = trigram_model.get(history, Counter())

        if not counter:
            fallback_history = (sentence[-1],)
            counter = bigram_model.get(fallback_history, Counter())

        if not counter:
            break

        next_word = choose_next_word(counter, temperature=temperature)

        if next_word == '</s>':
            break

        sentence.append(next_word)

    tokens = [token for token in sentence if token not in {'<s>', '</s>'}]
    if not tokens:
        return ''

    sentence_text = ' '.join(tokens)
    return sentence_text[0].upper() + sentence_text[1:] + '.'


generated_sentences = []
seen_sentences = set()

for _ in range(40):
    sentence = generate_sentence(max_length=28, temperature=0.12)
    if sentence and len(sentence.split()) >= 6 and sentence not in seen_sentences:
        generated_sentences.append(sentence)
        seen_sentences.add(sentence)

    if len(generated_sentences) >= 20:
        break

print('Generated Sentences:')
for index, sentence in enumerate(generated_sentences, start=1):
    print(f'{index}. {sentence}')

print('\nGenerated Paragraph:')
generated_text = ' '.join(generated_sentences)
print(generated_text)

output_path = notebook_dir / 'output.txt'
output_path.write_text(generated_text + '\n', encoding='utf-8')
print(f'\nSaved generated text to {output_path}')

Generated Sentences:
1. I have been so trying to convince them.
2. I have been brought to a very small part of the united state.
3. I have been unable to reach memphis will be no bloodshed or violence and there is no difference between u and the law of the united.
4. I have been unable to discern it.
5. I have been the addition of eight regiment of volunteer.
6. I have no purpose to let you see that you have a right to do so.
7. I have been beaten if some are satisfied and some have been so much of the united state.
8. I have been so much a listen to criminations among friend.

Generated Paragraph:
I have been so trying to convince them. I have been brought to a very small part of the united state. I have been unable to reach memphis will be no bloodshed or violence and there is no difference between u and the law of the united. I have been unable to discern it. I have been the addition of eight regiment of volunteer. I have no purpose to let you see that you have a right to do so. I h